# NeuroMera - Video Emotion Classification (ViT)

Extracts faces from video frames and classifies emotions using a Vision Transformer.

**How to use:**
1. GPU: `Runtime > Change runtime type > T4 GPU`
2. Organize your videos in labeled folders on Google Drive:
   ```
   VideoData/
     anger/
       clip1.mp4
       clip2.mp4
     joy/
       clip1.mp4
       ...
   ```
3. Update `DATASET_PATH` in Step 2
4. `Runtime > Run all`

---
## Step 1 - Install & Import

In [ ]:
!pip install -q transformers accelerate opencv-python-headless

In [ ]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from transformers import (
    ViTImageProcessor,
    ViTForImageClassification,
    TrainingArguments,
    Trainer,
)
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## Step 2 - Load Videos from Drive

In [ ]:
# ============================================================
#  CHANGE THIS to your video folder on Google Drive
#  Folder structure must be:
#    VideoData/anger/*.mp4, VideoData/joy/*.mp4, etc.
# ============================================================
DATASET_PATH  = "/content/drive/MyDrive/FYPVideo/VideoData"
MODEL_DIR     = "/content/drive/MyDrive/NeuroMera_Video_Model"
FRAMES_DIR    = "/content/frames"  # temporary, on Colab local disk
FRAMES_PER_VIDEO = 5               # frames to extract per video
# ============================================================

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".flv", ".wmv"}

videos = []
for label_dir in sorted(os.listdir(DATASET_PATH)):
    label_path = os.path.join(DATASET_PATH, label_dir)
    if not os.path.isdir(label_path):
        continue
    for fname in os.listdir(label_path):
        if os.path.splitext(fname)[1].lower() in VIDEO_EXTS:
            videos.append({"path": os.path.join(label_path, fname), "label": label_dir})

video_df = pd.DataFrame(videos)
print(f"Total videos: {len(video_df)}")
print(f"\nLabels found:\n{video_df['label'].value_counts()}")

---
## Step 3 - Extract Frames & Detect Faces

Extracts evenly-spaced frames from each video, detects the face, crops and saves it as a 224x224 image.

In [ ]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

def extract_faces_from_video(video_path, n_frames=5):
    """Extract n evenly-spaced frames, detect and crop face from each."""
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < 1:
        cap.release()
        return []

    indices = np.linspace(0, total - 1, n_frames, dtype=int)
    faces = []

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            continue

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        detected = face_cascade.detectMultiScale(gray, 1.3, 5)

        if len(detected) > 0:
            # Take the largest face
            x, y, w, h = max(detected, key=lambda r: r[2] * r[3])
            face = frame[y:y+h, x:x+w]
        else:
            # Fallback: center crop
            h, w = frame.shape[:2]
            size = min(h, w)
            y0 = (h - size) // 2
            x0 = (w - size) // 2
            face = frame[y0:y0+size, x0:x0+size]

        face = cv2.resize(face, (224, 224))
        face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
        faces.append(face)

    cap.release()
    return faces


# Extract frames and save to disk
os.makedirs(FRAMES_DIR, exist_ok=True)
frame_records = []

for i, row in video_df.iterrows():
    label = row["label"]
    label_dir = os.path.join(FRAMES_DIR, label)
    os.makedirs(label_dir, exist_ok=True)

    faces = extract_faces_from_video(row["path"], FRAMES_PER_VIDEO)
    video_id = os.path.splitext(os.path.basename(row["path"]))[0]

    for j, face in enumerate(faces):
        fname = f"{video_id}_f{j}.jpg"
        fpath = os.path.join(label_dir, fname)
        Image.fromarray(face).save(fpath)
        frame_records.append({"path": fpath, "label": label, "video_id": video_id})

    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(video_df)} videos...")

frame_df = pd.DataFrame(frame_records)
print(f"\nTotal face frames extracted: {len(frame_df)}")
print(f"\nPer label:\n{frame_df['label'].value_counts()}")

---
## Step 4 - Balance & Split

Split by **video** (not by frame) to prevent data leakage.

In [ ]:
# Balance: equal videos per emotion
min_videos = video_df["label"].value_counts().min()
balanced_videos = video_df.groupby("label", group_keys=False).apply(
    lambda x: x.sample(min_videos, random_state=42)
).reset_index(drop=True)

# Split by video ID
train_videos, test_videos = train_test_split(
    balanced_videos, test_size=0.2, stratify=balanced_videos["label"], random_state=42
)

train_video_ids = set(train_videos["path"].apply(
    lambda p: os.path.splitext(os.path.basename(p))[0]
))
test_video_ids = set(test_videos["path"].apply(
    lambda p: os.path.splitext(os.path.basename(p))[0]
))

# Assign frames to train/test based on which video they came from
train_frames = frame_df[frame_df["video_id"].isin(train_video_ids)].reset_index(drop=True)
test_frames  = frame_df[frame_df["video_id"].isin(test_video_ids)].reset_index(drop=True)

# Label mapping
labels_sorted = sorted(frame_df["label"].unique())
LABEL2ID = {label: i for i, label in enumerate(labels_sorted)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

print(f"Labels: {LABEL2ID}")
print(f"Train frames: {len(train_frames)} (from {len(train_video_ids)} videos)")
print(f"Test frames:  {len(test_frames)} (from {len(test_video_ids)} videos)")

---
## Step 5 - Dataset & Processor

In [ ]:
processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")

class FaceDataset(torch.utils.data.Dataset):
    def __init__(self, df, processor, label2id):
        self.df = df
        self.processor = processor
        self.label2id = label2id

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        inputs = self.processor(images=image, return_tensors="pt")
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        inputs["labels"] = self.label2id[row["label"]]
        return inputs


train_dataset = FaceDataset(train_frames, processor, LABEL2ID)
test_dataset  = FaceDataset(test_frames,  processor, LABEL2ID)

print(f"Datasets ready. Train: {len(train_dataset)}, Test: {len(test_dataset)}")

---
## Step 6 - Train ViT

Fine-tunes a pretrained Vision Transformer. Takes ~10-20 min on T4.

In [ ]:
model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=len(LABEL2ID),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)

def compute_metrics(pred):
    preds = pred.predictions.argmax(-1)
    return {"accuracy": accuracy_score(pred.label_ids, preds)}

training_args = TrainingArguments(
    output_dir="./video_results",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

---
## Step 7 - Evaluate

In [ ]:
preds_output = trainer.predict(test_dataset)
y_pred = preds_output.predictions.argmax(-1)
y_true = [test_dataset[i]["labels"] for i in range(len(test_dataset))]

print(classification_report(y_true, y_pred, target_names=list(LABEL2ID.keys())))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=list(LABEL2ID.keys()),
            yticklabels=list(LABEL2ID.keys()))
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

---
## Step 8 - Save Model to Google Drive

In [ ]:
model.save_pretrained(MODEL_DIR)
processor.save_pretrained(MODEL_DIR)

print(f"Model saved to {MODEL_DIR}")

---
## Step 9 - Load & Predict on a Video

Run from here to use an already-trained model without retraining.

In [ ]:
import os
import cv2
import torch
import numpy as np
from PIL import Image
from collections import Counter
from transformers import ViTForImageClassification, ViTImageProcessor

MODEL_DIR = "/content/drive/MyDrive/NeuroMera_Video_Model"

model = ViTForImageClassification.from_pretrained(MODEL_DIR)
processor = ViTImageProcessor.from_pretrained(MODEL_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)


def predict_video_emotion(video_path, n_frames=10):
    """Predict emotion from a video by majority vote across frames."""
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < 1:
        cap.release()
        return "error: could not read video"

    indices = np.linspace(0, total - 1, n_frames, dtype=int)
    predictions = []

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            continue

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        detected = face_cascade.detectMultiScale(gray, 1.3, 5)

        if len(detected) > 0:
            x, y, w, h = max(detected, key=lambda r: r[2] * r[3])
            face = frame[y:y+h, x:x+w]
        else:
            h, w = frame.shape[:2]
            size = min(h, w)
            y0 = (h - size) // 2
            x0 = (w - size) // 2
            face = frame[y0:y0+size, x0:x0+size]

        face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
        image = Image.fromarray(face)
        inputs = processor(images=image, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
        pred = torch.argmax(outputs.logits, dim=1).item()
        predictions.append(model.config.id2label[pred])

    cap.release()

    if not predictions:
        return "error: no frames processed"

    # Majority vote
    vote = Counter(predictions).most_common(1)[0][0]
    return vote


print("Model loaded and ready.")

In [ ]:
# Test on a video — change this path
test_video = "/content/drive/MyDrive/FYPVideo/test.mp4"  # CHANGE THIS

if os.path.exists(test_video):
    emotion = predict_video_emotion(test_video)
    print(f"Predicted Emotion: {emotion}")
else:
    print(f"File not found: {test_video}")
    print("Update test_video path to a .mp4 file on your Drive.")